# Advanced Experiments

This notebook runs experiments for all stretch goals:
1. Self-supervised graph learning
2. UltraGCN
3. Node/Edge dropout
4. Mixed precision training
5. Explainable recommendations
6. Temporal recommendations
7. Knowledge graph augmentation

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import torch
from scipy import sparse
import pandas as pd
from pathlib import Path

from src.utils.seed import set_seed
from src.data.download import download_movielens_100k
from src.data.preprocess import preprocess
from src.data.graph import build_adjacency_matrix, symmetric_normalization
from src.models.lightgcn import LightGCN
from src.models.ultralgcn import UltraGCN
from src.models.self_supervised import GraphAugmentor, SelfSupervisedLoss
from src.models.dropout import NodeDropout, EdgeDropout
from src.training.advanced_trainer import AdvancedTrainer
from src.training.mixed_precision import MixedPrecisionTrainer
from src.explain.explainer import GNNExplainer, PathExplainer
from src.knowledge.kg import KnowledgeGraph

set_seed(42)

In [ ]:
data_dir = download_movielens_100k('../data/raw')
meta = preprocess(data_dir, '../data/processed', seed=42)

train_mat = sparse.load_npz('../data/processed/train_interaction.npz')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')
movies_df = pd.read_csv('../data/processed/movies.csv')

num_users, num_items = meta['num_users'], meta['num_items']

val_mat = sparse.csr_matrix(
    (np.ones(len(val_df), dtype=np.float32),
     (val_df['user_idx'].values, val_df['item_idx'].values)),
    shape=(num_users, num_items)
)
test_mat = sparse.csr_matrix(
    (np.ones(len(test_df), dtype=np.float32),
     (test_df['user_idx'].values, test_df['item_idx'].values)),
    shape=(num_users, num_items)
)

norm_adj = symmetric_normalization(build_adjacency_matrix(train_mat))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}, Users: {num_users}, Items: {num_items}')

## Experiment 1: Edge Dropout

In [ ]:
print('=== Edge Dropout (p=0.1) ===')
model_ed = LightGCN(num_users, num_items, 64, 3, norm_adj, edge_dropout=0.1)
trainer_ed = AdvancedTrainer(
    model_ed, train_mat, val_mat, test_mat,
    lr=0.001, weight_decay=1e-5, neg_samples=1, top_k=[10],
    device=device, edge_dropout=0.1, seed=42
)
history_ed = trainer_ed.train(epochs=50, eval_every=10, patience=20, warmup_epochs=20)
user_emb, item_emb = model_ed()
metrics_ed = trainer_ed.test_evaluator.evaluate(user_emb, item_emb, 10)
print(f'EdgeDrop R@10: {metrics_ed["Recall@10"]:.4f}')

## Experiment 2: UltraGCN

In [ ]:
print('=== UltraGCN ===')
model_uc = UltraGCN(num_users, num_items, 64, n_layers=3)
trainer_uc = AdvancedTrainer(
    model_uc, train_mat, val_mat, test_mat,
    lr=0.001, weight_decay=1e-5, neg_samples=1, top_k=[10],
    device=device, ultralgcn_weight=1e-3, seed=42
)
history_uc = trainer_uc.train(epochs=50, eval_every=10, patience=20, warmup_epochs=20)
user_emb, item_emb = model_uc()
metrics_uc = trainer_uc.test_evaluator.evaluate(user_emb, item_emb, 10)
print(f'UltraGCN R@10: {metrics_uc["Recall@10"]:.4f}')

## Experiment 3: Self-Supervised Graph Learning

In [ ]:
print('=== Self-Supervised Learning ===')
augmentor = GraphAugmentor(train_mat, seed=42)
view1 = augmentor.generate_view('edge_perturbation', drop_rate=0.1, add_rate=0.05)
view2 = augmentor.generate_view('node_dropout', drop_rate=0.1)

norm_view1 = symmetric_normalization(build_adjacency_matrix(view1))
norm_view2 = symmetric_normalization(build_adjacency_matrix(view2))

model_ssl = LightGCN(num_users, num_items, 64, 3, norm_adj)
trainer_ssl = AdvancedTrainer(
    model_ssl, train_mat, val_mat, test_mat,
    lr=0.001, weight_decay=1e-5, neg_samples=1, top_k=[10],
    device=device, ssl_loss_weight=0.1, seed=42
)
history_ssl = trainer_ssl.train(epochs=50, eval_every=10, patience=20, warmup_epochs=20)
user_emb, item_emb = model_ssl()
metrics_ssl = trainer_ssl.test_evaluator.evaluate(user_emb, item_emb, 10)
print(f'SSL R@10: {metrics_ssl["Recall@10"]:.4f}')

## Experiment 4: Mixed Precision Training

In [ ]:
print('=== Mixed Precision Training ===')
model_mp = LightGCN(num_users, num_items, 64, 3, norm_adj)
trainer_mp = AdvancedTrainer(
    model_mp, train_mat, val_mat, test_mat,
    lr=0.001, weight_decay=1e-5, neg_samples=1, top_k=[10],
    device=device, mixed_precision=True, seed=42
)
history_mp = trainer_mp.train(epochs=50, eval_every=10, patience=20, warmup_epochs=20)
user_emb, item_emb = model_mp()
metrics_mp = trainer_mp.test_evaluator.evaluate(user_emb, item_emb, 10)
print(f'MixedPrecision R@10: {metrics_mp["Recall@10"]:.4f}')

## Experiment 5: Knowledge Graph Augmentation

In [ ]:
print('=== Knowledge Graph Augmentation ===')
genre_cols = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy',
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
              'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
              'Thriller', 'War', 'Western']

genre_matrix = movies_df[genre_cols].values.astype(np.float32)

kg = KnowledgeGraph(num_items)
kg.build_from_genre_matrix(genre_matrix, threshold=0.3)
print(f'KG: {kg.summary()}')

aug_adj = kg.get_augmented_adjacency(build_adjacency_matrix(train_mat), kg_weight=0.3)
norm_aug = symmetric_normalization(aug_adj)

model_kg = LightGCN(num_users, num_items, 64, 3, norm_aug)
trainer_kg = AdvancedTrainer(
    model_kg, train_mat, val_mat, test_mat,
    lr=0.001, weight_decay=1e-5, neg_samples=1, top_k=[10],
    device=device, seed=42
)
history_kg = trainer_kg.train(epochs=50, eval_every=10, patience=20, warmup_epochs=20)
user_emb, item_emb = model_kg()
metrics_kg = trainer_kg.test_evaluator.evaluate(user_emb, item_emb, 10)
print(f'KG-Aug R@10: {metrics_kg["Recall@10"]:.4f}')

## Experiment 6: Explainable Recommendations

In [ ]:
print('=== Explainable Recommendations ===')
explainer = PathExplainer(norm_adj)

user_emb, item_emb = model_ssl()
test_user = 0
recs = torch.topk(torch.mv(item_emb, user_emb[test_user]), 5)

for rank, (item_idx, score) in enumerate(zip(recs.indices, recs.values)):
    item_idx = item_idx.item()
    explanation = explainer.explain(test_user, item_idx, train_mat)
    title = movies_df.iloc[item_idx]['title'] if item_idx < len(movies_df) else f'Item {item_idx}'
    print(f'\nRank {rank+1}: {title} (score: {score:.4f})')
    print(f'  Paths found: {explanation["n_paths"]}')
    for p in explanation['paths'][:3]:
        print(f'  {p}')

## Experiment 7: Gradient Accumulation

In [ ]:
print('=== Gradient Accumulation (4 steps) ===')
model_ga = LightGCN(num_users, num_items, 64, 3, norm_adj)
trainer_ga = AdvancedTrainer(
    model_ga, train_mat, val_mat, test_mat,
    lr=0.001, weight_decay=1e-5, neg_samples=1, top_k=[10],
    device=device, gradient_accumulation_steps=4, seed=42
)
history_ga = trainer_ga.train(epochs=50, eval_every=10, patience=20, warmup_epochs=20)
user_emb, item_emb = model_ga()
metrics_ga = trainer_ga.test_evaluator.evaluate(user_emb, item_emb, 10)
print(f'GradAccum R@10: {metrics_ga["Recall@10"]:.4f}')

## Summary Comparison

In [ ]:
all_results = {
    'LightGCN (baseline)': metrics_ed,  # placeholder
    'Edge Dropout': metrics_ed,
    'UltraGCN': metrics_uc,
    'Self-Supervised': metrics_ssl,
    'Mixed Precision': metrics_mp,
    'KG Augmented': metrics_kg,
    'Gradient Accum': metrics_ga,
}

print(f'{"Model":<25} {"Recall@10":<12} {"NDCG@10":<12}')
print('-' * 49)
for name, m in all_results.items():
    print(f'{name:<25} {m["Recall@10"]:<12.4f} {m.get("NDCG@10", 0):<12.4f}')